In [1]:
# 1. Install the PubChem database connector
!pip install pubchempy

import pandas as pd
import pubchempy as pcp
import time

# 2. Load your exact ANOVA file
print("Loading ANOVA file...")
df = pd.read_csv('Removed duplicates ANOVA.csv')

# Extract the drug names from the first column
drugs = df.iloc[:, 0].dropna().astype(str).str.strip().unique()

smiles_list = []
failed_drugs = []

print(f"Starting conversion for {len(drugs)} drugs. This may take 2-3 minutes. Please wait...")

# 3. Query the database for each drug
for drug in drugs:
    try:
        # Search PubChem by drug name
        compounds = pcp.get_compounds(drug, 'name')
        if compounds:
            # Extract the SMILES string
            smiles = compounds[0].canonical_smiles
            # Format for SwissADME: SMILES [Space] DrugName
            smiles_list.append(f"{smiles} {drug}")
        else:
            failed_drugs.append(drug)
    except Exception as e:
        failed_drugs.append(drug)

    # Pause for a fraction of a second so we don't overload the PubChem server
    time.sleep(0.2)

print(f"\n✅ Successfully found SMILES for {len(smiles_list)} drugs.")
print(f"⚠️ Could not find SMILES for {len(failed_drugs)} experimental/numeric drugs.")

# 4. Split into batches of 100 and save for SwissADME
batch_size = 100
for i in range(0, len(smiles_list), batch_size):
    batch = smiles_list[i:i + batch_size]
    batch_num = (i // batch_size) + 1

    filename = f'SwissADME_Batch_{batch_num}.txt'
    with open(filename, 'w') as f:
        for item in batch:
            f.write(f"{item}\n")
    print(f"Saved {filename} containing {len(batch)} drugs.")

print("\nAll done! You can now download your Batch files from the Colab folder.")

Loading ANOVA file...
Starting conversion for 288 drugs. This may take 2-3 minutes. Please wait...


/tmp/ipykernel_322/3924344874.py:27: PubChemPyDeprecationWarning: canonical_smiles is deprecated: Use connectivity_smiles instead
  smiles = compounds[0].canonical_smiles



✅ Successfully found SMILES for 232 drugs.
⚠️ Could not find SMILES for 56 experimental/numeric drugs.
Saved SwissADME_Batch_1.txt containing 100 drugs.
Saved SwissADME_Batch_2.txt containing 100 drugs.
Saved SwissADME_Batch_3.txt containing 32 drugs.

All done! You can now download your Batch files from the Colab folder.
